<div style="background-color:#EAEAEA; padding:20px; border-left:5px solid #6C757D; border-radius:6px;">
<table style="width:100%; border:none;"><tr style="border:none;"><td style="border:none; vertical-align:top;">
<h1 style="font-size:32px; margin-top:0;">Master's Thesis</h1><hr style="margin:16px 0 22px 0;">
<p style="font-size:22px; line-height:1.5; margin:0;"><strong>Master's Degree in Advanced Physics</strong> - <strong>Universitat de València</strong></p>
<p style="font-size:17px; margin-top:28px; margin-bottom:6px;">This notebook is part of the <strong>Master's Thesis (MSc Dissertation)</strong>:</p>
<div style="font-size:25px; font-weight:700; line-height:1.3; margin-top:14px; margin-bottom:26px;">Fast Simulation of Neutrino Oscillations in Matter</div>
<p style="font-size:14px; line-height:1.55;"><strong>Author</strong><br>Juan Ramon Diaz Santos - <a href="mailto:diazjuan@alumni.uv.es">diazjuan@alumni.uv.es</a></p>
<p style="font-size:14px; line-height:1.55;"><strong>Supervisors</strong><br>Roberto Ruiz de Austri Bazan — <a href="mailto:rruiz@ific.uv.es">rruiz@ific.uv.es</a><br>Michele Lucente — <a href="mailto:michele.lucente@unibo.it">michele.lucente@unibo.it</a></p>
<p style="font-size:14px; line-height:1.55; margin-bottom:0;"><strong>Date</strong><br>September 2026</p></td>
<td style="border:none; width:230px; padding-left:25px; text-align:right; vertical-align:top;"><img src="../../../../logo_uv.png" alt="Universitat de València" style="width:200px; margin-top:5px;"></td>
</tr></table></div>

# Zenodo B23/SF-III Canonical Data Generator

Downloads the versioned Zenodo archive, verifies its MD5 checksum and generates only the products supplied or physically derivable from SF3. It does not import Bahcall spectra.

## 1. Imports and paths

In [1]:
from pathlib import Path
from io import BytesIO, StringIO
import hashlib
import json
import re
import urllib.request
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "data").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the TPeanuts project root")


def fetch(url: str, *, timeout: int = 60) -> bytes:
    request = urllib.request.Request(url, headers={"User-Agent": "TPeanuts/solar-data"})
    with urllib.request.urlopen(request, timeout=timeout) as response:
        return response.read()


def download(url: str, path: Path, *, overwrite: bool = False) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    if overwrite or not path.exists():
        path.write_bytes(fetch(url))
    return path


PROJECT_ROOT = find_project_root(Path.cwd())
SOLAR_DATA = PROJECT_ROOT / "data" / "solar"

## 2. Download and verify archive

In [2]:
PROVIDER_DIR = SOLAR_DATA / "zenodo"
for category in ("raw", "density", "production", "flux", "metadata"):
    (PROVIDER_DIR / category).mkdir(parents=True, exist_ok=True)
RECORD_API = "https://zenodo.org/api/records/10822316"
record = json.loads(fetch(RECORD_API))
file_info = record["files"][0]
archive_path = download(file_info["links"]["self"], PROVIDER_DIR / "raw" / file_info["key"])
expected_md5 = file_info["checksum"].removeprefix("md5:")
assert hashlib.md5(archive_path.read_bytes()).hexdigest() == expected_md5
with zipfile.ZipFile(archive_path) as bundle:
    for member in bundle.namelist():
        name = Path(member).name
        if name.startswith(("struct+nu_SF3_", "fluxes_SF3_")) and name.endswith(".dat"):
            (PROVIDER_DIR / "raw" / name).write_bytes(bundle.read(member))
print("Raw SF3 tables extracted.")

Raw SF3 tables extracted.


## 3. Generate density, radial production and flux products

In [3]:
MODELS = ("AAG21", "AGSS09", "C11", "GS98", "MB22m", "MB22p")
SOURCES = ("pp", "pep", "hep", "7Be", "8B", "13N", "15O", "17F")
AZ = {"H1": (1,1), "He4": (4,2), "He3": (3,2), "C12": (12,6), "C13": (13,6), "N14": (14,7), "N15": (15,7), "O16": (16,8), "O17": (17,8), "O18": (18,8), "Ne": (20,10), "Na": (23,11), "Mg": (24,12), "Al": (27,13), "Si": (28,14), "P": (31,15), "S": (32,16), "Cl": (35,17), "Ar": (36,18), "K": (39,19), "Ca": (40,20), "Sc": (45,21), "Ti": (48,22), "V": (51,23), "Cr": (52,24), "Mn": (55,25), "Fe": (56,26), "Co": (59,27), "Ni": (58,28)}
for model in MODELS:
    path = PROVIDER_DIR / "raw" / f"struct+nu_SF3_{model}.dat"
    columns = path.read_text(encoding="utf-8").splitlines()[0].split()[1:]
    table = pd.read_csv(path, sep=r"\s+", skiprows=1, names=columns)
    rho = 10.0 ** table["logRho"].to_numpy()
    ne = 10.0 ** table["log_ne"].to_numpy()
    nn = rho * sum(table[name].to_numpy() * (A-Z)/A for name, (A,Z) in AZ.items())
    pd.DataFrame({"radius": table["R_sun"], "electron_density_mol_cm3": ne, "neutron_density_mol_cm3": nn, "mass_density_g_cm3": rho}).to_csv(PROVIDER_DIR / "density" / f"density_SF3_{model}.csv", index=False)
    production = pd.DataFrame({"radius": table["R_sun"]})
    for source in SOURCES:
        production[f"{source} fraction"] = table[f"nu_{source}"]
    production.to_csv(PROVIDER_DIR / "production" / f"production_SF3_{model}.csv", index=False)
    lines = (PROVIDER_DIR / "raw" / f"fluxes_SF3_{model}.dat").read_text().splitlines()
    names, values = lines[1].split(), [float(value) for value in lines[2].split()[:8]]
    pd.DataFrame({"fraction": names, "flux": values}).to_csv(PROVIDER_DIR / "flux" / f"fluxes_SF3_{model}.csv", index=False)
(PROVIDER_DIR / "metadata" / "source.json").write_text(json.dumps({"provider": "B23 / SF-III", "doi": "10.5281/zenodo.10822316", "version": "v1.2", "default": True, "default_model": "SF3_AGSS09", "energy_spectra_available": False, "probabilities_available": False}, indent=2) + "\n")
print("Generated density, production and flux tables for", ", ".join(MODELS))

Generated density, production and flux tables for AAG21, AGSS09, C11, GS98, MB22m, MB22p


## 4. Validation

In [4]:
for model in MODELS:
    d = pd.read_csv(PROVIDER_DIR / "density" / f"density_SF3_{model}.csv")
    p = pd.read_csv(PROVIDER_DIR / "production" / f"production_SF3_{model}.csv")
    assert d["radius"].is_monotonic_increasing and (d.iloc[:, 1:] >= 0).all().all()
    assert set(f"{source} fraction" for source in SOURCES).issubset(p.columns)
print("Zenodo canonical products validated; spectrum/probability remain unavailable.")

Zenodo canonical products validated; spectrum/probability remain unavailable.
